# 02 - Building Spatial Labels

## Problem Solved by This Notebook
The spatial dataset does not include a supervised label ready to train the classifier. For that reason, this notebook builds an initial, interpretable, and reproducible `ground truth` based on hybrid rules.

## Methodological Strategy
1. Use real dataset percentiles to avoid invented thresholds.
2. Transform `UDI`, `SUN_HOURS`, and `RADIATION` into ordinal levels 1-5.
3. Combine those levels into an `exposure_score`.
4. Reserve `unsuitable` only for real extremes, not for normal low-light conditions.
5. Review class balance before moving to modeling.

## Output Classes
- `unsuitable`
- `shade_tolerant`
- `medium_indirect`
- `bright_indirect`
- `high_light`

### Which Percentiles Are Used Here and Why
This notebook does not reuse the entire exploratory list from notebook 01. Here, only `p20`, `p40`, `p60`, and `p80` are used.

The reason is operational and methodological:
- they divide each variable into 5 ordered levels of comparable size
- they keep the system easy to explain
- they avoid creating too many cut points that are difficult to defend
- they work well for combining `UDI`, `SUN_HOURS`, and `RADIATION` on the same ordinal scale

`p25`, `p50`, `p75`, and `p33`, `p67` are not used for the final labeling because the goal here is not classic quartiles or thirds, but five consistent exposure levels.


In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
GEOMETRY_PATH = RAW_DIR / 'geometry_dataset_daylight.csv'
df = pd.read_csv(GEOMETRY_PATH).reset_index(names='ROW_ID')
df.head()

## 1. Discretization Criterion
Each environmental variable is converted to a 1-5 scale. This makes it possible to mix variables of different magnitudes without losing the relative exposure order that exists within the dataset.

The equivalence is:
- `<= p20` -> level 1
- `p20-p40` -> level 2
- `p40-p60` -> level 3
- `p60-p80` -> level 4
- `> p80` -> level 5

This does not mean that level 5 is universally high in every project, but high within the distribution observed in this dataset.


In [ ]:
# These are the operational percentiles for the final labeling.
# They were chosen because they divide the sample into five comparable ordinal levels.
PERCENTILE_LEVELS = [0.2, 0.4, 0.6, 0.8]

def build_thresholds(series):
    # Percentiles make it possible to adapt the cut points to this specific dataset.
    return series.quantile(PERCENTILE_LEVELS).to_list()

def assign_level(value, thresholds):
    # Esta funcion preserva un orden simple: 1 = baja exposicion, 5 = alta exposicion.
    if value <= thresholds[0]:
        return 1
    if value <= thresholds[1]:
        return 2
    if value <= thresholds[2]:
        return 3
    if value <= thresholds[3]:
        return 4
    return 5

udi_thresholds = build_thresholds(df['UDI'])
sun_thresholds = build_thresholds(df['SUN_HOURS'])
rad_thresholds = build_thresholds(df['RADIATION'])

thresholds_df = pd.DataFrame({
    'UDI': udi_thresholds,
    'SUN_HOURS': sun_thresholds,
    'RADIATION': rad_thresholds
}, index=['p20', 'p40', 'p60', 'p80'])
display(thresholds_df)
thresholds_df.to_csv(PROCESSED_DIR / 'label_thresholds.csv')

### Reading the Thresholds
This table turns the exploration from notebook 01 into concrete cut points. It does not define absolute values for good or bad light, but relative bands within the dataset.

The important point here is that the jumps between percentiles are not uniform. This confirms that the three variables have different distributions and that the same numeric increase does not mean the same intensity of change across all ranges.

The methodological implication is direct: these thresholds were not chosen to look symmetric, but to respect the real shape of the data and make each cell's position comparable across `UDI`, `SUN_HOURS`, and `RADIATION`.


## 2. Building the Exposure Score
The score synthesizes three dimensions of exposure. `UDI` receives more weight because it better represents useful daylight quality. `RADIATION` keeps a high weight because it reflects solar load. `SUN_HOURS` weighs slightly less because it is a coarser accumulated measure.

The percentiles directly affect this step because they define levels 1-5 for each variable. In other words, the score is not calculated from raw values, but from a relative reading of each cell's position within the dataset.


In [ ]:
df['UDI_level'] = df['UDI'].apply(assign_level, thresholds=udi_thresholds)
df['SUN_level'] = df['SUN_HOURS'].apply(assign_level, thresholds=sun_thresholds)
df['RAD_level'] = df['RADIATION'].apply(assign_level, thresholds=rad_thresholds)

# The score combines the three variables into a single ordinal exposure reading.
df['exposure_score'] = (
    0.45 * df['UDI_level'] +
    0.20 * df['SUN_level'] +
    0.35 * df['RAD_level']
)

df[['TILE_ID', 'UDI_level', 'SUN_level', 'RAD_level', 'exposure_score']].head()

### Interpreting the Exposure Score
The `exposure_score` synthesizes three readings into a single ordered exposure measure. It does not replace the original variables, but it does allow them to be compared on a common scale.

Its correct reading is relative:
- low scores: cells closer to the low-exposure extreme
- medium scores: cells with intermediate behavior, useful for indirect-light categories
- high scores: clearly exposed cells, candidates for `bright_indirect` or `high_light`

This helps the later labeling avoid depending on a single variable and makes the classes emerge from a combined and more robust criterion.


## 3. Labeling Rules
The most important rule is that `unsuitable` does not simply mean low light. It is reserved for underexposure or overexposure extremes. The remaining cells fall into four classes that are useful for recommending species.


In [ ]:
def assign_spatial_label(row):
    # Underexposure extreme: the cell falls in the lowest level of the three variables.
    low_extreme = row['UDI_level'] == 1 and row['SUN_level'] == 1 and row['RAD_level'] == 1

    # Overexposure extreme: high radiation and many sun hours,
    # with UDI already clearly high.
    high_extreme = row['RAD_level'] == 5 and row['SUN_level'] == 5 and row['UDI_level'] >= 4

    if low_extreme or high_extreme:
        return 'unsuitable'

    score = row['exposure_score']
    if score < 2.2:
        return 'shade_tolerant'
    if score < 3.1:
        return 'medium_indirect'
    if score < 4.1:
        return 'bright_indirect'
    return 'high_light'

df['spatial_label'] = df.apply(assign_spatial_label, axis=1)
df[['TILE_ID', 'exposure_score', 'spatial_label']].head()

## 4. Class Balance Validation
Before training, it is mandatory to check whether the labeling produced classes with reasonable volume. If a class barely exists, the later model will not have a stable basis for learning it.


In [ ]:
label_counts = df['spatial_label'].value_counts().rename_axis('spatial_label').reset_index(name='count')
label_counts['pct'] = (label_counts['count'] / len(df) * 100).round(2)
display(label_counts)

plt.figure(figsize=(8, 4))
sns.barplot(data=label_counts, x='spatial_label', y='count', palette='YlGnBu')
plt.title('Balance inicial de clases espaciales')
plt.xticks(rotation=20)
plt.tight_layout()

label_counts.to_csv(PROCESSED_DIR / 'spatial_label_distribution.csv', index=False)

### Reading the Class Balance
This step validates whether the hybrid labeling produced a trainable problem. Having five class names is not enough; each class needs enough volume for the later classifier to learn it.

What is desirable here is:
1. that `unsuitable` remains an extreme class and does not absorb a large part of the dataset
2. that the intermediate classes have enough presence
3. that no class is so dominant that it makes the `accuracy` in notebook 05 misleading

If any class were too small or too large, this would be the moment to adjust thresholds or rules before training the model.


In [ ]:
# Este resumen permite interpretar como se ve cada clase en terminos reales
# of UDI, sun hours, and radiation.
label_summary = (
    df.groupby('spatial_label')[['UDI', 'SUN_HOURS', 'RADIATION', 'exposure_score']]
      .agg(['mean', 'min', 'max'])
      .round(2)
)
display(label_summary)
label_summary.to_csv(PROCESSED_DIR / 'spatial_label_summary.csv')

### What the Class Summary Validates
The aggregated table by class makes it possible to check that the labels are not arbitrary. If the means and ranges of `UDI`, `SUN_HOURS`, `RADIATION`, and `exposure_score` show a coherent order across classes, then the spatial taxonomy has internal consistency.

This strengthens the defense of the pipeline because it demonstrates that the labeling is not only interpretable in theory, but also observable in the aggregated data.


## 5. Notebook Output
The result is `geometry_labeled.csv`. This file becomes the supervised base of the project: it keeps the original information and adds levels, score, and spatial label.


In [ ]:
output_columns = [
    'ROW_ID', 'TILE_ID', 'BUILDING_ORIENTATION', 'RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W',
    'UDI', 'SUN_HOURS', 'RADIATION',
    'UDI_level', 'SUN_level', 'RAD_level', 'exposure_score', 'spatial_label'
]
df[output_columns].to_csv(PROCESSED_DIR / 'geometry_labeled.csv', index=False)
print('Saved:', PROCESSED_DIR / 'geometry_labeled.csv')